<a href="https://colab.research.google.com/github/RATKY07/Lenguajes_programacion/blob/main/lanzar_cripto_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🪙 Fondo Cripto Dashboard — Streamlit en Colab

Ejecuta las celdas **en orden** para lanzar el dashboard interactivo.

| Paso | Celda | Qué hace |
|------|-------|----------|
| 1 | Instalación | Instala dependencias |
| 2 | Crear app.py | Escribe el código del dashboard |
| 3 | Lanzar | Inicia Streamlit + túnel público |

In [1]:
# ─── CELDA 1: Instalar dependencias ───────────────────────────
!pip install streamlit yfinance plotly scipy -q
!npm install -g localtunnel -q
print('✅ Dependencias instaladas')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 85.6 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
added 22 packages in 3s
⠸
⠸3 packages are looking for funding
⠸  run `npm fund` for details
⠸✅ Dependencias instaladas


In [2]:
# ─── CELDA 2: Crear app.py ────────────────────────────────────
app_code = '''
import streamlit as st
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, date, timedelta
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

st.set_page_config(page_title="Cripto Dashboard", page_icon="🪙", layout="wide", initial_sidebar_state="expanded")

CRIPTOS_DEFAULT = ["BTC-USD", "ETH-USD", "SOL-USD", "ADA-USD", "LTC-USD"]
NOMBRES = {"BTC-USD": "Bitcoin", "ETH-USD": "Ethereum", "SOL-USD": "Solana", "LTC-USD": "Litecoin", "ADA-USD": "Cardano"}
COLORS  = {"BTC-USD": "#F7931A", "ETH-USD": "#627EEA", "SOL-USD": "#9945FF", "LTC-USD": "#F3BA2F", "ADA-USD": "#4A90D9"}
PORTFOLIO_COLORS = {"Conservador": "#1f77b4", "Moderado": "#ff7f0e", "Arriesgado": "#d62728"}

@st.cache_data(show_spinner=False)
def descargar_datos(criptos, inicio, fin):
    df_close = pd.DataFrame()
    for ticker in criptos:
        try:
            data = yf.download(ticker, start=inicio, end=fin, interval="1d", progress=False)
            if data.empty: continue
            if isinstance(data.columns, pd.MultiIndex):
                df_close[ticker] = data[("Close", ticker)]
            else:
                df_close[ticker] = data["Close"]
        except Exception as e:
            st.warning(f"Error {ticker}: {e}")
    return df_close

with st.sidebar:
    st.title("⚙️ Panel de Control")
    st.divider()
    st.subheader("🪙 Activos")
    criptos_sel = st.multiselect("Selecciona criptomonedas:", CRIPTOS_DEFAULT, default=CRIPTOS_DEFAULT, format_func=lambda x: f"{NOMBRES[x]} ({x})")
    if not criptos_sel:
        st.error("Selecciona al menos una criptomoneda.")
        st.stop()
    st.divider()
    st.subheader("📅 Rango de fechas")
    fecha_inicio = st.date_input("Fecha inicio:", value=date(2025, 1, 1), min_value=date(2018,1,1))
    fecha_fin    = st.date_input("Fecha fin:",    value=date.today(), min_value=fecha_inicio+timedelta(days=7))
    st.divider()
    st.subheader("📌 Benchmark")
    benchmark = st.selectbox("Referencia:", criptos_sel, format_func=lambda x: NOMBRES.get(x,x))
    st.divider()
    st.subheader("⏱️ Período")
    intervalo = st.radio("Retornos:", ["Diario","Semanal","Mensual"], horizontal=True)
    st.divider()
    st.button("🔄 Actualizar datos", use_container_width=True, type="primary")

with st.spinner("📡 Descargando datos..."):
    df_close = descargar_datos(tuple(criptos_sel), str(fecha_inicio), str(fecha_fin))

if df_close.empty:
    st.error("No se obtuvieron datos. Revisa el rango de fechas."); st.stop()

freq_map = {"Diario": None, "Semanal": "W", "Mensual": "ME"}
freq = freq_map[intervalo]
df_base    = df_close.resample(freq).last() if freq else df_close.copy()
df_returns = df_base.pct_change().dropna()
df_cum     = (1 + df_returns).cumprod() - 1
cov_matrix = df_returns.cov()
daily_avg  = df_returns.mean()
factor_anual = {"Diario":365,"Semanal":52,"Mensual":12}[intervalo]

st.title("🪙 Fondo de Criptomonedas — Dashboard Financiero")
st.markdown(f"**Período:** `{fecha_inicio}` → `{fecha_fin}` | **Intervalo:** `{intervalo}` | **Activos:** {', '.join(NOMBRES.get(c,c) for c in criptos_sel)}")
st.divider()

tab1, tab2, tab3, tab4, tab5 = st.tabs(["📈 Precios & Retornos","📉 Riesgo & Volatilidad","🔗 Correlación & Estadísticas","💼 Portafolios","🎛️ Análisis Personalizado"])

# ── TAB 1 ──────────────────────────────────────────────────────
with tab1:
    st.subheader("📈 Precios Históricos y Retornos")
    cols_kpi = st.columns(len(criptos_sel))
    for i, t in enumerate(criptos_sel):
        if t not in df_close.columns: continue
        pa = df_close[t].dropna().iloc[-1]; pi = df_close[t].dropna().iloc[0]
        cols_kpi[i].metric(NOMBRES.get(t,t), f"${pa:,.2f}", f"{(pa/pi-1)*100:+.1f}%")
    st.divider()
    c1,c2 = st.columns(2)
    modo_precio = c1.radio("Vista:", ["Precio absoluto (USD)","Precio normalizado (base 100)"], horizontal=True)
    escala_y    = c2.radio("Escala Y:", ["Lineal","Logarítmica"], horizontal=True)
    fig_p = go.Figure()
    for t in criptos_sel:
        if t not in df_close.columns: continue
        s = df_close[t].dropna()
        y = (s/s.iloc[0])*100 if "normalizado" in modo_precio else s
        fig_p.add_trace(go.Scatter(x=s.index,y=y,mode="lines",name=NOMBRES.get(t,t),line=dict(color=COLORS.get(t,"#FFF"),width=2)))
    fig_p.update_layout(title="Precios Históricos",yaxis_type="log" if escala_y=="Logarítmica" else "linear",template="plotly_dark",hovermode="x unified",height=420)
    st.plotly_chart(fig_p, use_container_width=True)
    fig_cum2 = go.Figure()
    for t in criptos_sel:
        if t not in df_cum.columns: continue
        fig_cum2.add_trace(go.Scatter(x=df_cum.index,y=df_cum[t],mode="lines",name=NOMBRES.get(t,t),line=dict(color=COLORS.get(t,"#FFF"),width=2)))
    fig_cum2.update_layout(title=f"Retorno Acumulado ({intervalo})",yaxis_tickformat=".1%",template="plotly_dark",hovermode="x unified",height=380)
    st.plotly_chart(fig_cum2, use_container_width=True)
    fig_ret2 = go.Figure()
    for t in criptos_sel:
        if t not in df_returns.columns: continue
        fig_ret2.add_trace(go.Scatter(x=df_returns.index,y=df_returns[t],mode="lines",name=NOMBRES.get(t,t),line=dict(color=COLORS.get(t,"#FFF"),width=1.5)))
    fig_ret2.update_layout(title=f"Retornos {intervalo}s",yaxis_tickformat=".1%",template="plotly_dark",hovermode="x unified",height=350)
    st.plotly_chart(fig_ret2, use_container_width=True)

# ── TAB 2 ──────────────────────────────────────────────────────
with tab2:
    st.subheader("📉 Riesgo y Volatilidad")
    disponibles = [t for t in criptos_sel if t in df_returns.columns]
    std_devs = df_returns[disponibles].std().reset_index()
    std_devs.columns=["Cripto","Desv"]; std_devs["Nombre"]=std_devs["Cripto"].map(NOMBRES)
    fig_std2=px.bar(std_devs,x="Nombre",y="Desv",color="Cripto",color_discrete_map=COLORS,text_auto=".4f",template="plotly_dark",title="Desviación Estándar")
    fig_std2.update_traces(textposition="outside"); st.plotly_chart(fig_std2,use_container_width=True)
    max_dd={}; dd_ser={}
    for t in disponibles:
        cum=(1+df_returns[t]).cumprod(); rm=cum.expanding().max(); dd=(rm-cum)/rm
        max_dd[t]=dd.max(); dd_ser[t]=dd
    df_dd2=pd.DataFrame(max_dd.items(),columns=["Cripto","MaxDD"]); df_dd2["Nombre"]=df_dd2["Cripto"].map(NOMBRES)
    fig_dd2=px.bar(df_dd2,x="Nombre",y="MaxDD",color="Cripto",color_discrete_map=COLORS,text_auto=".2%",template="plotly_dark",title="Máxima Caída (Max Drawdown)")
    fig_dd2.update_layout(yaxis_tickformat=".1%"); fig_dd2.update_traces(textposition="outside"); st.plotly_chart(fig_dd2,use_container_width=True)
    lf=(df_returns[disponibles]<0).sum()/len(df_returns); df_lf2=lf.reset_index()
    df_lf2.columns=["Cripto","Freq"]; df_lf2["Nombre"]=df_lf2["Cripto"].map(NOMBRES)
    fig_lf2=px.bar(df_lf2,x="Nombre",y="Freq",color="Cripto",color_discrete_map=COLORS,text_auto=".1%",template="plotly_dark",title="Frecuencia de Pérdidas")
    fig_lf2.update_layout(yaxis_tickformat=".1%"); fig_lf2.update_traces(textposition="outside"); st.plotly_chart(fig_lf2,use_container_width=True)
    risk_df2=pd.DataFrame({"Criptomoneda":[NOMBRES.get(t,t) for t in disponibles],"Vol Anualizada":[df_returns[t].std()*np.sqrt(factor_anual) for t in disponibles],"Max Drawdown":[max_dd.get(t,np.nan) for t in disponibles],"% Períodos neg":[lf.get(t,np.nan) for t in disponibles]}).set_index("Criptomoneda")
    st.dataframe(risk_df2.style.format({"Vol Anualizada":"{:.2%}","Max Drawdown":"{:.2%}","% Períodos neg":"{:.1%}"}).background_gradient(cmap="RdYlGn_r"),use_container_width=True)

# ── TAB 3 ──────────────────────────────────────────────────────
with tab3:
    st.subheader("🔗 Correlación y Estadísticas")
    disponibles=[ t for t in criptos_sel if t in df_returns.columns]
    if len(disponibles)>=2:
        corr=df_returns[disponibles].corr()
        corr.index=corr.columns=[NOMBRES.get(t,t) for t in disponibles]
        fig_corr2=px.imshow(corr,text_auto=".2f",color_continuous_scale="RdBu_r",zmin=-1,zmax=1,aspect="auto",title="Matriz de Correlación",template="plotly_dark")
        fig_corr2.update_layout(height=420); st.plotly_chart(fig_corr2,use_container_width=True)
    td=st.selectbox("Analizar distribución de:",disponibles,format_func=lambda x:NOMBRES.get(x,x))
    if td in df_returns.columns:
        rs=df_returns[td].dropna(); bins2=st.slider("Bins:",10,100,40)
        fg=go.Figure()
        fg.add_trace(go.Histogram(x=rs,nbinsx=bins2,name="Retornos",marker_color=COLORS.get(td,"#627EEA"),opacity=0.8))
        xr=np.linspace(rs.min(),rs.max(),200); yn=stats.norm.pdf(xr,rs.mean(),rs.std())*len(rs)*(rs.max()-rs.min())/bins2
        fg.add_trace(go.Scatter(x=xr,y=yn,mode="lines",name="Normal teórica",line=dict(color="white",dash="dash",width=2)))
        fg.update_layout(title=f"Distribución — {NOMBRES.get(td,td)}",xaxis_tickformat=".1%",template="plotly_dark",height=380)
        st.plotly_chart(fg,use_container_width=True)
        _,pn=stats.normaltest(rs)
        cm1,cm2,cm3,cm4=st.columns(4)
        cm1.metric("Media",f"{rs.mean():.4f}"); cm2.metric("Std Dev",f"{rs.std():.4f}")
        cm3.metric("Curtosis",f"{float(rs.kurtosis()):.2f}"); cm4.metric("p-valor normalidad",f"{pn:.4f}")

# ── TAB 4 ──────────────────────────────────────────────────────
with tab4:
    st.subheader("💼 Portafolios")
    disponibles=[t for t in criptos_sel if t in df_returns.columns]
    _all_w={"Conservador":{"BTC-USD":0.60,"ETH-USD":0.20,"LTC-USD":0.15,"SOL-USD":0.025,"ADA-USD":0.025},
            "Moderado":{"BTC-USD":0.40,"ETH-USD":0.30,"LTC-USD":0.15,"SOL-USD":0.075,"ADA-USD":0.075},
            "Arriesgado":{"BTC-USD":0.20,"ETH-USD":0.25,"LTC-USD":0.10,"SOL-USD":0.225,"ADA-USD":0.225}}
    def norm_w(pw,disp):
        sub={k:v for k,v in pw.items() if k in disp}; tot=sum(sub.values())
        return {k:v/tot for k,v in sub.items()} if tot>0 else sub
    portfolios={n:norm_w(p,disponibles) for n,p in _all_w.items()}
    def met_p(w):
        tk=list(w.keys()); wv=np.array([w[t] for t in tk])
        rd=sum(daily_avg.get(t,0)*w[t] for t in tk); ra=rd*factor_anual
        sc=cov_matrix.loc[tk,tk]; var=np.dot(wv.T,np.dot(sc,wv)); va=np.sqrt(var)*np.sqrt(factor_anual)
        return {"Retorno":ra,"Volatilidad":va,"Sharpe":ra/va if va>0 else 0}
    pc=st.columns(3)
    for i,(n,pw) in enumerate(portfolios.items()):
        m=met_p(pw)
        with pc[i]:
            st.markdown(f"**{n}**")
            st.metric("Retorno anual",f"{m['Retorno']:.2%}"); st.metric("Volatilidad",f"{m['Volatilidad']:.2%}"); st.metric("Sharpe",f"{m['Sharpe']:.3f}")
    st.divider()
    ps=st.selectbox("Ver portafolio:",list(portfolios.keys()))
    pw2=portfolios[ps]
    c_p,c_c=st.columns(2)
    with c_p:
        dpie=pd.DataFrame([{"Criptomoneda":NOMBRES.get(t,t),"Peso":w,"Ticker":t} for t,w in pw2.items()])
        fp=px.pie(dpie,values="Peso",names="Criptomoneda",color="Ticker",color_discrete_map={t:COLORS.get(t,"#FFF") for t in pw2},hole=0.35,title=f"Portafolio {ps}",template="plotly_dark")
        fp.update_traces(textinfo="percent+label"); st.plotly_chart(fp,use_container_width=True)
    with c_c:
        fpc2=go.Figure()
        for n,pw3 in portfolios.items():
            tk3=[t for t in pw3 if t in df_returns.columns]
            ws3=pd.Series({t:pw3[t] for t in tk3}); rp3=df_returns[tk3].dot(ws3); cp3=(1+rp3).cumprod()-1
            fpc2.add_trace(go.Scatter(x=cp3.index,y=cp3,mode="lines",name=n,line=dict(color=PORTFOLIO_COLORS[n],width=2)))
        if benchmark in df_cum.columns:
            fpc2.add_trace(go.Scatter(x=df_cum.index,y=df_cum[benchmark],mode="lines",name=f"{NOMBRES.get(benchmark,benchmark)} (Benchmark)",line=dict(color=COLORS.get(benchmark,"#FFF"),dash="dash",width=1.5)))
        fpc2.update_layout(title="Portafolios vs. Benchmark",yaxis_tickformat=".1%",template="plotly_dark",hovermode="x unified",height=380)
        st.plotly_chart(fpc2,use_container_width=True)
    st.divider()
    st.markdown("#### 🎛️ Portafolio Personalizado")
    pesos_c={}; sc2=st.columns(len(disponibles))
    for i,t in enumerate(disponibles):
        with sc2[i]: pesos_c[t]=st.slider(NOMBRES.get(t,t),0,100,int(100/len(disponibles)),key=f"sl_{t}")
    tp=sum(pesos_c.values())
    if tp==0: st.error("Pesos no pueden ser 0")
    else:
        pn2={t:w/tp for t,w in pesos_c.items()}
        st.success(f"Suma: {tp}%") if abs(tp-100)<=1 else st.warning(f"Suma: {tp}% → se normalizará")
        mc=met_p(pn2); mc1,mc2,mc3=st.columns(3)
        mc1.metric("Retorno anual",f"{mc['Retorno']:.2%}"); mc2.metric("Volatilidad",f"{mc['Volatilidad']:.2%}"); mc3.metric("Sharpe",f"{mc['Sharpe']:.3f}")
        tk_c=[t for t in pn2 if t in df_returns.columns]; ws_c=pd.Series({t:pn2[t] for t in tk_c})
        rc=(df_returns[tk_c].dot(ws_c)); cc=(1+rc).cumprod()-1
        fc2=go.Figure(); fc2.add_trace(go.Scatter(x=cc.index,y=cc,mode="lines",name="Mi portafolio",line=dict(color="#00ff88",width=2.5)))
        if benchmark in df_cum.columns: fc2.add_trace(go.Scatter(x=df_cum.index,y=df_cum[benchmark],mode="lines",name="Benchmark",line=dict(color=COLORS.get(benchmark,"#FFF"),dash="dash",width=1.5)))
        fc2.update_layout(title="Mi Portafolio vs. Benchmark",yaxis_tickformat=".1%",template="plotly_dark",hovermode="x unified",height=350)
        st.plotly_chart(fc2,use_container_width=True)

# ── TAB 5 ──────────────────────────────────────────────────────
with tab5:
    st.subheader("🎛️ Análisis Personalizado")
    disponibles=[t for t in criptos_sel if t in df_returns.columns]
    if len(disponibles)>=2:
        ca1,ca2=st.columns(2)
        aa=ca1.selectbox("Activo A:",disponibles,format_func=lambda x:NOMBRES.get(x,x),key="aa")
        ab=ca2.selectbox("Activo B:",[t for t in disponibles if t!=aa],format_func=lambda x:NOMBRES.get(x,x),key="ab")
        fcomp=make_subplots(rows=2,cols=1,shared_xaxes=True,subplot_titles=["Precio normalizado","Retornos"])
        for t in [aa,ab]:
            s=df_close[t].dropna(); fcomp.add_trace(go.Scatter(x=s.index,y=(s/s.iloc[0])*100,mode="lines",name=NOMBRES.get(t,t),line=dict(color=COLORS.get(t,"#FFF"),width=2)),row=1,col=1)
            fcomp.add_trace(go.Scatter(x=df_returns.index,y=df_returns[t],mode="lines",name=NOMBRES.get(t,t),line=dict(color=COLORS.get(t,"#FFF"),width=1.5),showlegend=False),row=2,col=1)
        fcomp.update_layout(template="plotly_dark",height=500,hovermode="x unified"); fcomp.update_yaxes(tickformat=".1%",row=2,col=1)
        st.plotly_chart(fcomp,use_container_width=True)
        corr_ab=df_returns[aa].corr(df_returns[ab]); st.metric(f"Correlación {NOMBRES.get(aa,aa)} ↔ {NOMBRES.get(ab,ab)}",f"{corr_ab:.4f}")
    st.divider()
    st.markdown("#### 💰 Simulador de Inversión")
    cs1,cs2=st.columns(2)
    mi=cs1.number_input("Monto inicial (USD):",min_value=100.0,max_value=1000000.0,value=10000.0,step=100.0)
    ci=cs1.selectbox("Cripto:",disponibles,format_func=lambda x:NOMBRES.get(x,x),key="ci")
    est=cs2.radio("Estrategia:",["Comprar y mantener","DCA (aporte mensual)"])
    am=cs2.number_input("Aporte mensual:",min_value=0.0,value=500.0,step=50.0) if est=="DCA (aporte mensual)" else 0
    if ci in df_close.columns:
        pr=df_close[ci].dropna()
        if est=="Comprar y mantener":
            vf=mi*(pr.iloc[-1]/pr.iloc[0]); pv=(pr/pr.iloc[0])*mi; ti=mi
        else:
            pm=pr.resample("ME").last(); un=0.0; ti=0.0; vt=[]
            for f,p in pm.items(): un+=am/p; ti+=am; vt.append((f,un*p,ti))
            vf=vt[-1][1] if vt else mi; ti=vt[-1][2] if vt else mi
            pv=pd.Series({v[0]:v[1] for v in vt}) if vt else pr/pr.iloc[0]*mi
        ss1,ss2,ss3=st.columns(3)
        ss1.metric("Valor final",f"${vf:,.2f}",f"${vf-ti:+,.2f}")
        ss2.metric("Retorno total",f"{(vf/ti-1)*100:+.1f}%"); ss3.metric("Invertido",f"${ti:,.2f}")
        fs=go.Figure(); fs.add_trace(go.Scatter(x=pv.index,y=pv,mode="lines",name="Valor",line=dict(color=COLORS.get(ci,"#00ff88"),width=2),fill="tozeroy"))
        fs.update_layout(title=f"Simulación — {NOMBRES.get(ci,ci)}",yaxis_tickprefix="$",template="plotly_dark",height=350); st.plotly_chart(fs,use_container_width=True)
    st.divider()
    st.markdown("#### 📥 Exportar")
    e1,e2=st.columns(2)
    e1.download_button("⬇️ Precios CSV",df_close.to_csv().encode(),"precios_cripto.csv","text/csv",use_container_width=True)
    e2.download_button("⬇️ Retornos CSV",df_returns.to_csv().encode(),"retornos_cripto.csv","text/csv",use_container_width=True)

st.divider()
st.markdown("<div style=\'text-align:center;color:#666;font-size:0.85em;\'>🪙 Fondo Cripto Dashboard · Yahoo Finance · Streamlit + Plotly</div>",unsafe_allow_html=True)
'''

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(app_code.strip())
print('✅ app.py creado exitosamente')

✅ app.py creado exitosamente


In [3]:
# ─── CELDA 3: Lanzar Streamlit + túnel público ────────────────
import subprocess, threading, time

def run_streamlit():
    subprocess.run(['streamlit', 'run', 'app.py',
                    '--server.port', '8501',
                    '--server.headless', 'true',
                    '--server.enableCORS', 'false'])

t = threading.Thread(target=run_streamlit, daemon=True)
t.start()
time.sleep(4)  # esperar que arranque

# Obtener URL pública
import urllib.request
url = urllib.request.urlopen('https://loca.lt/mytunnelpassword').read().decode()
print(f'🔑 Tunnel password: {url}')

proc = subprocess.Popen(
    ['lt', '--port', '8501'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
for line in proc.stdout:
    line = line.decode().strip()
    if 'loca.lt' in line:
        print(f'\n🚀 Dashboard disponible en: {line}')
        print(f'🔑 Si pide contraseña, ingresa: {url}')
        break

🔑 Tunnel password: 34.73.198.99

🚀 Dashboard disponible en: your url is: https://all-impalas-glow.loca.lt
🔑 Si pide contraseña, ingresa: 34.73.198.99
